# Quante scuole primarie statali hanno un locale mensa?

Un confronto tra le province italiane, anno scolastico 2024/25

## Introduzione

Nelle settimane scorse il dibattito sulla scuola ha riguardato soprattutto il tetto del 30% agli alunni stranieri nelle classi prime e il divieto di volto coperto, previsti dal decreto-legge approvato dal Consiglio dei ministri il 24 settembre 2026.

A prescindere dal giudizio personale su queste misure, io ho ritenuto importante focalizzarmi su un aspetto più concreto: come sono fatti gli edifici in cui i bambini vanno a scuola.

In particolare, in questa analisi, partendo dai dati del Ministero dell'istruzione e del merito (MIM), mi sono chiesto:

**Quante scuole primarie statali hanno un locale mensa, e quanto cambia da provincia a provincia?**

**Attenzione: i dati dicono se l'edificio ha un *locale* adibito a mensa, non se i bambini mangiano a scuola. La differenza è spiegata nei limiti dell'analisi, in fondo.**

Far luce sulle situazioni reali nel nostro Paese è fondamentale sia per indirizzare le risorse pubbliche agli interventi effettivamente necessari, sia per mostrare ai non addetti qual è la situazione del nostro paese. Inoltre, fondamentale è diffondere la necessità di avere dati aperti, utilizzabili da tutti così da favorire analisi utili a migliorare la vita di tutti noi.

## Dati utilizzati e premesse

I dati vengono dal portale Open Data del Ministero dell'istruzione e del merito (licenza IODL 2.0), anno scolastico 2024/25, aggiornati al 6 agosto 2025. Ho usato due dataset sulle scuole statali:

- **Elenco e localizzazione degli edifici** (anagrafe): in quale comune e provincia si trova ogni edificio;
- **Destinazione d'uso degli spazi** (dotazioni): per ogni edificio, se ha aule, mensa, palestra, auditorium e altri spazi.

**Chi compila i dati.** Le informazioni sono fornite dagli enti locali proprietari o gestori degli edifici (per le primarie, in genere i comuni), ai sensi della legge 11 gennaio 1996, n. 23. Il portale avverte inoltre che sono ancora in corso gli aggiornamenti dell'associazione tra edifici e scuole. Non posso quindi escludere informazioni mancanti o incomplete.

**Come sono scritti i valori.** Per ogni edificio e per ogni spazio il dato può essere:

- `SI`: l'edificio ha quello spazio;
- `NO`: l'edificio non ha quello spazio;
- `NON DEFINITO`: l'informazione non è disponibile (un campo vuoto viene trattato allo stesso modo).

**Perché la mensa.** Non tutte le dotazioni sono utilizzabili: per aule, aule informatiche e piscina circa il 26% dei valori è `NON DEFINITO`. Per mensa, palestra e auditorium la quota è molto bassa (circa lo 0,6% degli edifici). Tra le dotazioni utilizzabili ho scelto la mensa, e come grado la scuola primaria, per il legame con il tempo pieno.
:**Cosa è escluso.**

- Le scuole paritarie.
- Le province autonome di Trento e Bolzano, che non sono nel dataset.
- La Valle d'Aosta: i codici delle sue scuole hanno una struttura diversa (il terzo e il quarto carattere sono `1E`, `1A`, `1M` invece di `EE`, `AA`, `MM`) e il grado non si ricava allo stesso modo. Ho scelto di escluderla: sono circa 80 plessi con codice `1E`, che probabilmente sono le primarie.

## Importazione dei dati

Leggo tutti i campi come testo (`dtype='str'`), così non si perdono gli zeri iniziali dei codici comune (ad esempio `084039`). Disattivo anche la conversione automatica in valore mancante: altrimenti pandas legge la sigla `NA` (Napoli) come dato vuoto.

In [5]:
# importazione delle librerie necessarie
import pandas as pd

In [6]:
# importazione del 2 file di origine
#Pandas riconosce la provincia di Napoli (NA) come valore nullo quindi
#inseriamo nella lettura dei file keep_default_na=False, na_values=['']

# File 1 anagrafica delle scuole e degli edifici
df_anagrafe = pd.read_csv('EDIANAGRAFESTA202120242520250806.csv', dtype = 'str', keep_default_na=False, na_values=[''])

# File 2 dotazioni degli edifici
df_dotazioni = pd.read_csv('EDIAMBFUNZSTA202120242520250806.csv', dtype = 'str', keep_default_na=False, na_values=[''])

In [7]:
# visualizzazione del contenuto del File 1
df_anagrafe.head()

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,CODICECOMUNE,DESCRIZIONECOMUNE,SIGLAPROVINCIA,TIPOLOGIAINDIRIZZO,DENOMINAZIONEINDIRIZZO,NUMEROCIVICO,CAP
0,202425,FRTF00601R,0600101813,060010,ARPINO,FR,Via,PELAGALLI,3,03033
1,202425,VTEE81201A,0560240021,056024,FABRICA DI ROMA,VT,Via,Della Repubblica,3,01034
2,202425,UDPC010005,0301290059,030129,UDINE,UD,Piazza,I° Maggio,26,33100
3,202425,CSAA89604L,0780450908,078045,COSENZA,CS,Via,Giulia,SNC,87100
4,202425,RIEE817095,0570550153,057055,POGGIO NATIVO,RI,Via,Archipiglione,snc,02030


In [8]:
# visualizzazione del contenuto del File 2
df_dotazioni.head()

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,SPAZIDIDATTICA,AULE,AULEINFORMATICHE,AULETECNICHE,SPAZICOLLETTIVI,AUDITORIUM,MENSA,PALESTRA,PISCINA,SPAZIAMMINISTRAZIONE
0,202425,BOMM84201T,0370320944,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO
1,202425,NUEE87002E,0910310241,SI,SI,SI,SI,SI,NO,SI,NO,NO,NO
2,202425,PAIC87200Q,0820534498,SI,SI,NO,NO,SI,SI,NO,SI,NO,SI
3,202425,VRAA873023,0230722554,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO
4,202425,GRAA81701G,0530080202,SI,SI,NO,NO,NO,NO,NO,NO,NO,NO


## Pulizia dei dati

I passaggi sono tre:

1. verifica ed eliminazione dei duplicati, in ciascun file;
2. unione dei due file sulla coppia *codice scuola + codice edificio*;
3. verifica che tutte le righe siano state abbinate.

**Duplicati.** In ciascun file ci sono 4 coppie scuola-edificio ripetute: due scuole di Aosta, una di Gradisca d'Isonzo e una di Assemini. Nel caso controllato (Gradisca d'Isonzo) le righe sono identiche. Ho tenuto la prima occorrenza.

**Unione.** Dopo la pulizia i file hanno 60.026 righe ciascuno e si abbinano tutte: nessuna riga resta senza corrispondenza.

In [10]:
# verifico la presenza di duplicati nel file 1
df_anagrafe[df_anagrafe.duplicated(subset=['CODICESCUOLA','CODICEEDIFICIO'])]

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,CODICECOMUNE,DESCRIZIONECOMUNE,SIGLAPROVINCIA,TIPOLOGIAINDIRIZZO,DENOMINAZIONEINDIRIZZO,NUMEROCIVICO,CAP
4399,202425,GOAA81601L,0310080003,031008,GRADISCA D'ISONZO,GO,Via,Zucchelli,17,34072
22128,202425,AORR01000A,0070030616,007003,AOSTA,AO,Viale,Federico Chabod,8,11100
33493,202425,AOPM010003,0070030001,007003,AOSTA,AO,Via,Chavanne,23/E,11100
59917,202425,CAEE8AH01X,0920030948,092003,ASSEMINI,CA,Via,DI VITTORIO,19,09032


In [11]:
# ad esempio verifichiamo il primo duplicato filtrando per CODICESCUOLA
df_anagrafe[df_anagrafe['CODICESCUOLA'] == 'GOAA81601L']

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,CODICECOMUNE,DESCRIZIONECOMUNE,SIGLAPROVINCIA,TIPOLOGIAINDIRIZZO,DENOMINAZIONEINDIRIZZO,NUMEROCIVICO,CAP
2130,202425,GOAA81601L,0310080003,031008,GRADISCA D'ISONZO,GO,Via,Zucchelli,17,34072
4399,202425,GOAA81601L,0310080003,031008,GRADISCA D'ISONZO,GO,Via,Zucchelli,17,34072


In [12]:
# Elimino i duplicati
df_anagrafe = df_anagrafe.drop_duplicates(subset=['CODICESCUOLA','CODICEEDIFICIO'])

In [13]:
# verifico la presenza di duplicati nel file 2
df_dotazioni[df_dotazioni.duplicated(subset=['CODICESCUOLA','CODICEEDIFICIO'])]

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,SPAZIDIDATTICA,AULE,AULEINFORMATICHE,AULETECNICHE,SPAZICOLLETTIVI,AUDITORIUM,MENSA,PALESTRA,PISCINA,SPAZIAMMINISTRAZIONE
14292,202425,AORR01000A,0070030616,SI,SI,SI,SI,SI,SI,NO,SI,NO,SI
25720,202425,AOPM010003,0070030001,SI,SI,SI,SI,SI,SI,NO,SI,NO,SI
38185,202425,GOAA81601L,0310080003,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO
45995,202425,CAEE8AH01X,0920030948,SI,SI,SI,SI,NO,NO,NO,NO,NO,SI


In [14]:
# ad esempio verifichiamo il primo duplicato filtrando per CODICESCUOLA
df_dotazioni[df_dotazioni['CODICESCUOLA'] == 'GOAA81601L']

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,SPAZIDIDATTICA,AULE,AULEINFORMATICHE,AULETECNICHE,SPAZICOLLETTIVI,AUDITORIUM,MENSA,PALESTRA,PISCINA,SPAZIAMMINISTRAZIONE
37467,202425,GOAA81601L,0310080003,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO
38185,202425,GOAA81601L,0310080003,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO


In [15]:
# Elimino i duplicati
df_dotazioni = df_dotazioni.drop_duplicates(subset=['CODICESCUOLA','CODICEEDIFICIO'])

In [16]:
# Unisco i due file utilizzando come codice la coppia scuola+edificio
df = pd.merge(df_dotazioni, df_anagrafe, how = 'outer', on = ['ANNOSCOLASTICO','CODICESCUOLA','CODICEEDIFICIO'])

In [17]:
# Verifico se i match sono andati tutti in porto
d = pd.merge(df_dotazioni, df_anagrafe, how='outer',
             on=['ANNOSCOLASTICO','CODICESCUOLA','CODICEEDIFICIO'],
             indicator=True)
d['_merge'].value_counts()

_merge
both          60026
left_only         0
right_only        0
Name: count, dtype: int64

In [18]:
#Visualizzo il risultato finale
df.head()

,ANNOSCOLASTICO,CODICESCUOLA,CODICEEDIFICIO,SPAZIDIDATTICA,AULE,AULEINFORMATICHE,AULETECNICHE,SPAZICOLLETTIVI,AUDITORIUM,MENSA,PALESTRA,PISCINA,SPAZIAMMINISTRAZIONE,CODICECOMUNE,DESCRIZIONECOMUNE,SIGLAPROVINCIA,TIPOLOGIAINDIRIZZO,DENOMINAZIONEINDIRIZZO,NUMEROCIVICO,CAP
0,202425,BOMM84201T,0370320944,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO,037032,IMOLA,BO,Via,Luigi Pirandello,12-Ed. P,40026
1,202425,NUEE87002E,0910310241,SI,SI,SI,SI,SI,NO,SI,NO,NO,NO,091031,GIRASOLE,NU,Via,NAZIONALE,2,08040
2,202425,PAIC87200Q,0820534498,SI,SI,NO,NO,SI,SI,NO,SI,NO,SI,082053,PALERMO,PA,Via,F. PANZERA,28,90124
3,202425,VRAA873023,0230722554,SI,SI,NO,NO,SI,NO,SI,NO,NO,NO,023072,SANGUINETTO,VR,Via,S.Meritani,9,37058
4,202425,GRAA81701G,0530080202,SI,SI,NO,NO,NO,NO,NO,NO,NO,NO,053008,CIVITELLA PAGANICO,GR,Via,Grosseto,4,58045


## Dal file per edificio al file per scuola

Nei dati una *scuola* è un plesso, identificato dal `CODICESCUOLA`, e può occupare più edifici. Per un bambino conta avere accesso a una mensa, non in quale edificio sia: per questo una scuola risulta con mensa se **almeno un** suo edificio ce l'ha. È una scelta mia, non una regola dei dati.

La regola di attribuzione è questa:

- `SI`: almeno un edificio ha la mensa;
- `NO`: nessun edificio ha `SI` e almeno uno ha `NO`;
- `ND` (non definito): in tutti gli altri casi, cioè quando tutti gli edifici sono `NON DEFINITO` o vuoti.

I passaggi sono quattro:

1. una funzione che applica la regola;
2. una tabella con una riga per scuola e cinque colonne: `mensa`, `n_edifici`, `comune`, `descr_comune`, `provincia`;
3. una colonna `grado`, ricavata dal terzo e dal quarto carattere del codice scuola: `AA` infanzia, `EE` primaria, `MM` media. Le superiori hanno codici diversi a seconda dell'indirizzo, la Valle d'Aosta una struttura propria (vedi premesse);
4. un filtro sulle sole primarie (`EE`).

Le scuole con valore `ND` restano nel conteggio.

In [20]:
#creo la funzione che assegna SI se esiste almeno un edificio della scuola con la mensa, no se sono tutti indefiniti ma almeno uno con NO e ND se in tutti gli edifici risulta Non definito
def dotato(s):
    if (s == 'SI').any(): return 'SI'
    if (s == 'NO').any(): return 'NO'
    return 'ND'

In [21]:
#creo un dataset con una riga per ogni scuola, se dotato o meno di Palestra, mensa, con il numero di edifici il comune e la provincia
scuole = df.groupby('CODICESCUOLA').agg(
    mensa=('MENSA', dotato),
    n_edifici=('CODICEEDIFICIO', 'nunique'),
    comune=('CODICECOMUNE', 'first'),
    descr_comune=('DESCRIZIONECOMUNE', 'first'),
    provincia=('SIGLAPROVINCIA', 'first'),
).reset_index()

In [22]:
scuole

,CODICESCUOLA,mensa,n_edifici,comune,descr_comune,provincia
0,AGAA805032,SI,1,084039,SANT'ANGELO MUXARO,AG
1,AGAA805043,NO,1,084030,RAFFADALI,AG
2,AGAA805054,SI,1,084030,RAFFADALI,AG
3,AGAA80803D,SI,1,084038,SANTA MARGHERITA DI BELICE,AG
4,AGAA80804E,SI,1,084034,SAMBUCA DI SICILIA,AG
...,...,...,...,...,...,...
47802,VVTH01000A,NO,2,102027,PIZZO,VV
47803,VVTL01101X,NO,1,102047,VIBO VALENTIA,VV
47804,VVTL011519,NO,1,102047,VIBO VALENTIA,VV
47805,VVVC02000G,SI,1,102047,VIBO VALENTIA,VV


In [23]:
#verifico quanti CODICESCUOLA sono scritti in minuscolo
(scuole['CODICESCUOLA'] != scuole['CODICESCUOLA'].str.upper()).sum()

1

In [24]:
#trasformo i codici in minuscolo in tutto maiuscolo
scuole['CODICESCUOLA'] = scuole['CODICESCUOLA'].str.upper().str.strip()

In [25]:
scuole

,CODICESCUOLA,mensa,n_edifici,comune,descr_comune,provincia
0,AGAA805032,SI,1,084039,SANT'ANGELO MUXARO,AG
1,AGAA805043,NO,1,084030,RAFFADALI,AG
2,AGAA805054,SI,1,084030,RAFFADALI,AG
3,AGAA80803D,SI,1,084038,SANTA MARGHERITA DI BELICE,AG
4,AGAA80804E,SI,1,084034,SAMBUCA DI SICILIA,AG
...,...,...,...,...,...,...
47802,VVTH01000A,NO,2,102027,PIZZO,VV
47803,VVTL01101X,NO,1,102047,VIBO VALENTIA,VV
47804,VVTL011519,NO,1,102047,VIBO VALENTIA,VV
47805,VVVC02000G,SI,1,102047,VIBO VALENTIA,VV


In [26]:
#creo una colonna che mi indica il grado della scuola, la terza e la quarta lettera del codice scuola:
# AA = infanzia, EE = Primaria, MM = Media e le superiori hanno codici diversi a seconda dell'indirizzo.
scuole['grado'] = scuole['CODICESCUOLA'].str[2:4]

In [27]:
#controllo che valori ci sono all'interno della colonna grado
scuole['grado'].value_counts().head(10)

grado
EE    14513
AA    12852
MM     7139
IC     4645
IS     1386
TD     1146
PS     1001
TF      744
RI      646
CT      550
Name: count, dtype: int64

In [28]:
#filtro il dataset solo per le scuole primarie
primarie = scuole[scuole['grado'] == 'EE']

In [29]:
primarie

,CODICESCUOLA,mensa,n_edifici,comune,descr_comune,provincia,grado
100,AGEE805015,NO,1,084039,SANT'ANGELO MUXARO,AG,EE
101,AGEE80801L,SI,1,084038,SANTA MARGHERITA DI BELICE,AG,EE
102,AGEE80802N,NO,1,084025,MONTEVAGO,AG,EE
103,AGEE80803P,NO,1,084034,SAMBUCA DI SICILIA,AG,EE
104,AGEE81001L,NO,1,084020,LAMPEDUSA E LINOSA,AG,EE
...,...,...,...,...,...,...,...
47703,VVEE835081,SI,1,102005,CAPISTRANO,VV,EE
47704,VVEE835092,SI,1,102035,SAN NICOLA DA CRISSA,VV,EE
47705,VVEE83601D,NO,1,102019,LIMBADI,VV,EE
47706,VVEE83602E,NO,1,102019,LIMBADI,VV,EE


## Analisi

1. Conteggio delle primarie statali con e senza locale mensa e percentuali sul totale. Il perimetro è tutta Italia, esclusi Valle d'Aosta e province di Trento e Bolzano.
2. Tabella per provincia: numero di primarie, numero di primarie senza locale mensa, numero di scuole con dato non definito e percentuale di primarie senza locale mensa, calcolata su tutte le primarie della provincia.

In [31]:
#conto quante scuole hanno la mensa, quante non ce l'hanno e quante hanno un valore NON DEFINITO nel dataset
primarie['mensa'].value_counts()

mensa
SI    7928
NO    6556
ND      29
Name: count, dtype: int64

In [32]:
#calcoliamo la quota di ciacuna
primarie['mensa'].value_counts(normalize=True).mul(100).round(1)

mensa
SI    54.6
NO    45.2
ND     0.2
Name: proportion, dtype: float64

In [33]:
#costruisco la tabella per provincia con il numero di primarie, quelle senza mensa (e la %), quelle con valore mensa ND
per_provincia = primarie.groupby('provincia').agg(
    n_primarie=('CODICESCUOLA', 'size'),
    senza_mensa=('mensa', lambda s: (s == 'NO').sum()),
    mensa_nd=('mensa', lambda s: (s == 'ND').sum()),
).reset_index()

per_provincia['quota_senza_mensa'] = (per_provincia['senza_mensa'] / per_provincia['n_primarie'] * 100).round(1)

per_provincia.sort_values('quota_senza_mensa', ascending=False).head(10)

,provincia,n_primarie,senza_mensa,mensa_nd,quota_senza_mensa
75,RG,61,59,0,96.7
26,CT,262,228,1,87.0
55,NA,630,546,5,86.7
0,AG,94,77,0,81.9
59,PA,231,189,0,81.8
45,LT,130,105,0,80.8
91,TP,103,80,0,77.7
84,SR,74,57,0,77.0
73,RC,199,152,0,76.4
89,TE,89,65,0,73.0


### Risultati

Su 14.513 primarie statali, 7.928 (54,6%) hanno un locale mensa, 6.556 (45,2%) no, per 29 (0,2%) il dato non è definito.

Tra le province, la quota di primarie senza locale mensa va dal 2% circa di Prato a quasi il 97% di Ragusa. Le più alte sono Ragusa (96,7%), Catania (87,0%), Napoli (86,7%), Agrigento (81,9%) e Palermo (81,8%).

## Controlli

Ragusa ha il valore più alto (96,7%). Prima di usarlo ho verificato tre cose:

1. se Ragusa risulta `NO` su tutti gli spazi, cosa che farebbe pensare a un errore di compilazione;
2. se il valore è coerente con le altre province siciliane;
3. se dipende da qualche comune in particolare.

Il primo controllo riguarda **tutti gli edifici** della provincia, di ogni grado; gli altri due solo le primarie.

In [36]:
# 1. Ragusa è "NO" su tutto, o solo sulla mensa?
rg = df[df['SIGLAPROVINCIA'] == 'RG']
rg[['MENSA','PALESTRA','AUDITORIUM']].apply(lambda c: c.value_counts(normalize=True)).round(2)

,MENSA,PALESTRA,AUDITORIUM
NO,0.96,0.71,0.75
SI,0.04,0.29,0.25


In [37]:
# 2. Confronto con le province vicine della stessa regione
per_provincia[per_provincia['provincia'].isin(['RG','SR','CT','CL','EN','ME','PA','TP','AG'])]

,provincia,n_primarie,senza_mensa,mensa_nd,quota_senza_mensa
0,AG,94,77,0,81.9
21,CL,50,27,0,54.0
26,CT,262,228,1,87.0
28,EN,34,22,0,64.7
49,ME,245,155,0,63.3
59,PA,231,189,0,81.8
75,RG,61,59,0,96.7
84,SR,74,57,0,77.0
91,TP,103,80,0,77.7


In [38]:
#verifica i comuni in provincia di RG per vedere se c'è un problema di outlier - spoiler non c'è
primarie[primarie['provincia'] == 'RG'].groupby('descr_comune')['mensa'].value_counts().unstack(fill_value=0)

mensa,NO,SI
descr_comune,,
ACATE,2,0
CHIARAMONTE GULFI,3,0
COMISO,4,0
GIARRATANA,1,0
ISPICA,4,0
MODICA,15,0
MONTEROSSO ALMO,1,0
POZZALLO,2,1
RAGUSA,9,1


### Cosa emerge dai controlli

1. **Non è `NO` su tutto.** Tra gli edifici di Ragusa il 96% non ha la mensa, ma il 71% non ha la palestra e il 75% l'auditorium: chi ha compilato ha distinto tra i vari spazi.
2. **È la più alta, ma non è fuori scala.** Le province siciliane vanno dal 54,0% (Caltanissetta) al 96,7% (Ragusa); Catania è all'87,0%.
3. **Non dipende da un comune.** 59 primarie su 61 risultano senza locale mensa, in tutti i 12 comuni della provincia (le due eccezioni sono a Pozzallo e a Ragusa).
4. **Riscontro esterno.** Secondo elaborazioni di Save the Children e dell'Osservatorio sui conti pubblici su dati del Ministero (a.s. 2021/22), a Palermo, Ragusa e Siracusa solo il 6-8% degli alunni della primaria usa la mensa. Misure e anni diversi, stessa direzione.

**Cosa non ho verificato:** il confronto con gli anni scolastici precedenti. Non posso quindi escludere del tutto un errore di compilazione, e per questo presento i dati come *dichiarati* dagli enti locali.

## Limiti dell'analisi

- **Locale, non servizio.** Il dato dice se l'edificio ha uno spazio per la mensa, non se i bambini mangiano a scuola: una scuola può servire i pasti senza un locale dedicato. Le due misure vanno nella stessa direzione ma non coincidono: secondo l'Osservatorio sui conti pubblici il 39% delle primarie ha la mensa, mentre il 57% degli alunni usa il servizio.
- **Dati dichiarati dagli enti locali**, con possibili errori e omissioni.
- **Si contano scuole, non alunni.** Una provincia con molte piccole scuole senza mensa pesa più di una con poche scuole grandi.
- **Perimetro.** Solo scuole statali; esclusi Valle d'Aosta, Trento e Bolzano.
- **Province piccole.** Le primarie per provincia vanno da 34 (Enna) a 630 (Napoli): nelle più piccole un solo plesso sposta la percentuale di circa tre punti.
- **Una fotografia.** I dati si riferiscono all'anno scolastico 2024/25.

## Fonti

- MIM, Portale Unico dei Dati della Scuola, area Edilizia scolastica: https://dati.istruzione.it/opendata/opendata/catalogo/elements1/?area=Edilizia+Scolastica
- Decreto scuola, Sky TG24: https://tg24.sky.it/politica/2026/09/24/decreto-scuola-tetto-studenti-stranieri-cdm-oggi
- Decreto scuola, Focus Scuola: https://www.focus-scuola.it/decreto-scuola-valditara-tetto-del-30-burqa-e-latino/
- Save the Children, mensa e tempo pieno: https://www.savethechildren.it/blog-notizie/scuola-poco-piu-di-1-bambino-su-2-ha-accesso-al-servizio-mensa
- Osservatorio sui conti pubblici (OCPI), PNRR, tempo pieno e mense: https://osservatoriocpi.unicatt.it/ocpi-pubblicazioni-il-pnrr-l-estensione-del-tempo-pieno-e-le-mense-scolastiche-nelle-scuole-primarie

## Esportazione del file csv definitivo per tools di visualizzazione dei dati

In [43]:
export = per_provincia[['provincia', 'n_primarie', 'senza_mensa', 'quota_senza_mensa']].copy()

export.columns = ['Provincia', 'Primarie statali', 'Senza locale mensa (n)', 'Senza locale mensa (%)']

In [44]:
export.shape        # deve essere (104, 5)
export.head()

,Provincia,Primarie statali,Senza locale mensa (n),Senza locale mensa (%)
0,AG,94,77,81.9
1,AL,138,26,18.8
2,AN,114,37,32.5
3,AP,57,23,40.4
4,AQ,92,52,56.5


In [45]:
export.to_csv('mensa_primarie_province.csv', index=False)